In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from tqdm import tqdm

import predictive_coding as pc

# Reproducibility: same random numbers every run
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Paths are relative to the notebook, which lives in RELAX/notebooks/.
# data/ and runs/ are both listed in .gitignore, so nothing here gets committed.
DATA_DIR = Path("../data")
RUN_DIR = Path("../runs/mnist_bpc_generation")
RUN_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                      # PIL image -> tensor in [0, 1], shape (1, 28, 28)
    transforms.Lambda(lambda x: torch.flatten(x)),  # -> shape (784,)
])

train_dataset = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

batch_size = 500
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(len(train_dataset), len(test_dataset))

In [ ]:
class BidirectionalPC(nn.Module):
    def __init__(self, alpha_up=1.0, alpha_down=1.0):
        super().__init__()
        # The two latent layers x0 and x1. Each is used by BOTH pathways.
        self.pcs = nn.ModuleList([pc.PCLayer(), pc.PCLayer()])
        act = nn.ReLU()

        # bottom-up: image -> x0 -> x1 -> label
        self.model_up = nn.Sequential(
            nn.Linear(784, 256),
            self.pcs[0],
            act,
            nn.Linear(256, 256),
            self.pcs[1],
            act,
            nn.Linear(256, 10),
        )
        # top-down: label -> x1 -> x0 -> image   (note: pcs in reverse order)
        self.model_down = nn.Sequential(
            nn.Linear(10, 256),
            self.pcs[1],
            act,
            nn.Linear(256, 256),
            self.pcs[0],
            act,
            nn.Linear(256, 784),
        )

        self.alpha_up = alpha_up
        self.alpha_down = alpha_down

    def forward(self, inputs):
        img, y = inputs
        # 1) Bottom-up pass. Each PCLayer stores the energy 0.5*(x - mu_up)^2 in ._energy
        out_up = self.model_up(img)
        e_up = [p._energy for p in self.pcs]
        # 2) Top-down pass. It OVERWRITES ._energy with 0.5*(x - mu_down)^2
        out_down = self.model_down(y)
        e_down = [p._energy for p in self.pcs]
        # 3) Store the weighted sum, which is what PCTrainer will read
        for p, eu, ed in zip(self.pcs, e_up, e_down):
            p._energy = self.alpha_up * eu + self.alpha_down * ed
        return out_up, out_down


def loss_fn(outputs, _y, _img, alpha_up, alpha_down):
    """Energy of the two *output* terms: 0.5||y - y_hat||^2 and 0.5||u - u_hat||^2."""
    out_up, out_down = outputs
    return (alpha_up * 0.5 * (out_up - _y).pow(2).sum()
            + alpha_down * 0.5 * (out_down - _img).pow(2).sum())

In [ ]:
alpha_up, alpha_down = 1.0, 1.0
model = BidirectionalPC(alpha_up, alpha_down).to(device)
print(model)
print("weights:", sum(p.numel() for p in model.parameters()))

In [ ]:
T = 20  # inference steps per batch

trainer = pc.PCTrainer(
    model,
    T=T,
    optimizer_x_fn=optim.SGD, optimizer_x_kwargs={"lr": 0.01},   # inference
    update_p_at="last",
    optimizer_p_fn=optim.Adam, optimizer_p_kwargs={"lr": 1e-3},  # learning
    plot_progress_at=[],
)


def test_accuracy(model):
    """Classify with one bottom-up feed-forward pass (PCLayers transparent in eval mode)."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for img, label in test_loader:
            pred = model.model_up(img.to(device)).argmax(dim=1)
            correct += (pred == label.to(device)).sum().item()
    model.train()
    return correct / len(test_dataset)

In [ ]:
epochs = 5
model.train()  # PCLayers must be in train mode while training (the trainer asserts this)

accs = [test_accuracy(model)]
for epoch in range(epochs):
    for img, label in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        img = img.to(device)
        y = F.one_hot(label, num_classes=10).float().to(device)
        trainer.train_on_batch(
            inputs=(img, y),
            loss_fn=loss_fn,
            loss_fn_kwargs={"_y": y, "_img": img, "alpha_up": alpha_up, "alpha_down": alpha_down},
            is_log_progress=False,
            is_return_results_every_t=False,
            is_checking_after_callback_after_t=False,
        )
    accs.append(test_accuracy(model))
    print(f"epoch {epoch + 1}: test accuracy {accs[-1]:.4f}")

plt.plot(accs, marker="o")
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.grid(alpha=0.3)
plt.show()

In [ ]:
def weights_only(state_dict):
    """Drop the PCLayer latents (keys ending in '._x') from a state_dict."""
    return {k: v for k, v in state_dict.items() if not k.endswith("._x")}

ckpt_path = RUN_DIR / f"bpc_up{alpha_up}_down{alpha_down}.pt"
torch.save(weights_only(model.state_dict()), ckpt_path)
print("saved", ckpt_path)

In [ ]:
gen_model = BidirectionalPC(alpha_up, alpha_down).to(device)
print(gen_model.load_state_dict(torch.load(ckpt_path)))

# Freeze the weights: during generation only the latents and the image change.
for p in gen_model.parameters():
    p.requires_grad_(False)

In [ ]:
def one_hot(digits):
    return F.one_hot(torch.as_tensor(digits), num_classes=10).float().to(device)


def generate(model, schedule, lr_x=0.02, image_init="topdown", latent_init="topdown",
             record_every=40):
    """
    Run PC inference with the label clamped and the image free.

    schedule:    list of (digits, n_steps), e.g. [([0..9], 300), ([1..9,0], 10000)]
    image_init:  "topdown"  -> image starts as the top-down picture of the first label
                 "noise"    -> image starts as uniform noise
    latent_init: "topdown"  -> latents start as top-down predictions from the first label
                 "bottomup" -> latents start as bottom-up predictions from the starting image
    Returns frames (list of (batch, 784) tensors), the clamped label of each frame,
    and the energy at every step.
    """
    model.train()  # PCLayers must use their latents _x, not be transparent
    y0 = one_hot(schedule[0][0])
    batch = y0.shape[0]

    # (a) Where the IMAGE starts. It becomes a free variable: an nn.Parameter
    #     that the optimiser can update.
    if image_init == "topdown":
        model.eval()                          # transparent PCLayers: a plain top-down pass
        with torch.no_grad():
            img0 = model.model_down(y0)       # notebook 5's picture of the first label
        model.train()
    elif image_init == "noise":
        img0 = torch.rand(batch, 784, device=device)
    else:
        raise ValueError(f"unknown image_init: {image_init}")
    img = nn.Parameter(img0.clone())

    # (b) Where the LATENTS start: ONE forward pass with sampling switched on.
    #     set_is_sample_x(True) makes each PCLayer set _x := mu on the next forward call.
    #     This comes after (a) because bottom-up latents are computed FROM the image.
    for p in model.pcs:
        p.set_is_sample_x(True)
    if latent_init == "topdown":
        model.model_down(y0)                  # x1, x0 := predictions from the label
    elif latent_init == "bottomup":
        model.model_up(img)                   # x0, x1 := predictions from the starting image
    else:
        raise ValueError(f"unknown latent_init: {latent_init}")

    # (c) Inference optimiser: plain gradient descent on x0, x1 and u. The weights are frozen.
    opt = optim.SGD([p._x for p in model.pcs] + [img], lr=lr_x)

    frames, frame_labels, energies = [img.detach().clone()], [schedule[0][0]], []
    step = 0
    for digits, n_steps in schedule:
        y = one_hot(digits)  # <- the clamp. Changing this is the "swap".
        for _ in range(n_steps):
            opt.zero_grad()
            out_up, out_down = model((img, y))
            E = sum(p._energy for p in model.pcs) + loss_fn(
                (out_up, out_down), y, img, model.alpha_up, model.alpha_down)
            E.backward()   # dE/dx0, dE/dx1, dE/du  (the gradient from Section 1)
            opt.step()     # x <- x - lr * dE/dx
            energies.append(E.item())
            step += 1
            if step % record_every == 0:
                frames.append(img.detach().clone())
                frame_labels.append(digits)
    return frames, frame_labels, energies

In [ ]:
record_every = 40
A = list(range(10))
B = [(d + 1) % 10 for d in A]
schedule = [(A, 6_000), (B, 10_000)]   # from noise, A needs ~6000 steps to form before the swap

frames, frame_labels, energies = generate(gen_model, schedule, lr_x=0.02, image_init="noise",
                                          latent_init="bottomup", record_every=record_every)
print(len(frames), "frames")

plt.semilogy(energies)
plt.axvline(schedule[0][1], color="r", ls="--", label="label swap")
plt.xlabel("inference step"); plt.ylabel("energy E"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
W = dict(gen_model.named_parameters())
for name in ["model_up.0.weight", "model_up.3.weight", "model_up.6.weight",
             "model_down.0.weight", "model_down.3.weight", "model_down.6.weight"]:
    s = torch.linalg.matrix_norm(W[name], ord=2).item()
    print(f"{name:22s} sigma_max^2 = {s * s:6.1f}")

In [ ]:
def show_snapshots(frames, frame_labels, record_every, n_rows=8):
    idx = np.unique(np.geomspace(1, len(frames) - 1, n_rows - 1).astype(int))
    idx = [0] + list(idx)
    fig, axes = plt.subplots(len(idx), 10, figsize=(10, 1.1 * len(idx)))
    for r, k in enumerate(idx):
        for c in range(10):
            ax = axes[r, c]
            ax.imshow(frames[k][c].view(28, 28).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"{frame_labels[0][c]}→{frame_labels[-1][c]}", fontsize=8)
        axes[r, 0].set_ylabel(f"t={k * record_every}", fontsize=7)
    plt.tight_layout(); plt.show()

show_snapshots(frames, frame_labels, record_every)

In [ ]:
def classify(model, images):
    """Feed-forward classification of a (batch, 784) tensor; returns a list of labels."""
    model.eval()   # transparent PCLayers -> feed-forward classifier
    with torch.no_grad():
        pred = model.model_up(images).argmax(1).tolist()
    model.train()  # back to latent mode
    return pred

before = classify(gen_model, frames[schedule[0][1] // record_every])  # last frame before the swap
after = classify(gen_model, frames[-1])
print("before swap:", before)
print("after  swap:", after)

In [ ]:
# Print the predicted classes every 10 frames and flag the moments where any of them changes
prev = None
for i in range(0, len(frames), 10):
    pred = classify(gen_model, frames[i])
    flag = "  <- changed" if prev is not None and pred != prev else ""
    print(f"step {i * record_every:6d}: {pred}{flag}")
    prev = pred

In [ ]:
def save_gif(frames, frame_labels, path, record_every, fps=20):
    fig, axes = plt.subplots(1, 10, figsize=(10, 1.7))
    ims, titles = [], []
    for c, ax in enumerate(axes):
        ims.append(ax.imshow(frames[0][c].view(28, 28).clamp(0, 1), cmap="gray", vmin=0, vmax=1))
        titles.append(ax.set_title("", fontsize=9))
        ax.axis("off")
    suptitle = fig.suptitle("")

    def update(k):
        for c in range(10):
            ims[c].set_data(frames[k][c].view(28, 28).clamp(0, 1))
            titles[c].set_text(f"clamped: {frame_labels[k][c]}")
        suptitle.set_text(f"step {k * record_every}")
        return ims

    FuncAnimation(fig, update, frames=len(frames)).save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)

gif_path = RUN_DIR / "morph_d_to_d+1.gif"
save_gif(frames, frame_labels, gif_path, record_every)
print("saved", gif_path)

## Single digit through a sequence of clamps

Generate **one** image and clamp a list of labels on it, one after the other: e.g. `[1, 8, 4]` clamps 1, then 8, then 4.
The GIF shows the image next to the energy curve; dashed lines mark each clamp switch and the orange cursor is the current step.

This reuses `generate` unchanged, with a batch of one: the schedule becomes `[([1], 300), ([8], 5000), ([4], 5000)]`.

In [ ]:
def make_single_digit_gif(model, digits, steps_per_clamp, lr_x=0.02,
                          image_init="topdown", latent_init="topdown",
                          record_every=20, fps=25, path=None):
    """
    Generate ONE image and clamp a sequence of labels on it, one after the other.

    digits:          sequence of clamped labels, e.g. [1, 8, 4]
    steps_per_clamp: inference steps for each clamp, either one int (same for all)
                     or a list with one entry per digit, e.g. [300, 5000, 5000]
    image_init, latent_init: passed to generate (see its docstring)
    Saves a GIF with the image on the left and the energy on the right, and
    returns (frames, frame_labels, energies, path).
    """
    if isinstance(steps_per_clamp, int):
        steps_per_clamp = [steps_per_clamp] * len(digits)
    assert len(steps_per_clamp) == len(digits), "need one step count per digit"

    # A batch of ONE: every phase clamps a list with a single label, e.g. ([8], 5000)
    schedule = [([d], n) for d, n in zip(digits, steps_per_clamp)]
    frames, frame_labels, energies = generate(model, schedule, lr_x=lr_x,
                                              image_init=image_init, latent_init=latent_init,
                                              record_every=record_every)

    # step at which each new clamp starts, to mark the switches on the energy plot
    switch_steps = np.cumsum([0] + steps_per_clamp)[1:-1]

    fig, (ax_img, ax_E) = plt.subplots(1, 2, figsize=(8, 3.6),
                                       gridspec_kw={"width_ratios": [1, 1.6]})
    im = ax_img.imshow(frames[0][0].view(28, 28).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax_img.axis("off")

    ax_E.semilogy(energies, color="#2a78d6", lw=1)
    for s in switch_steps:
        ax_E.axvline(s, color="#52514e", ls="--", lw=1)
    cursor = ax_E.axvline(0, color="#eb6834", lw=2)       # moves with the animation
    ax_E.set_xlabel("inference step"); ax_E.set_ylabel("energy E")
    ax_E.set_title(f"image init: {image_init}   latent init: {latent_init}", fontsize=8)
    ax_E.grid(alpha=0.3)
    title = fig.suptitle("")
    fig.tight_layout()

    def update(k):
        step = k * record_every
        im.set_data(frames[k][0].view(28, 28).clamp(0, 1))
        cursor.set_xdata([step, step])
        title.set_text(f"clamped label: {frame_labels[k][0]}    step {step}")
        return im, cursor, title

    if path is None:
        # include both init choices so different runs don't overwrite each other
        path = RUN_DIR / (f"single_{'-'.join(map(str, digits))}"
                          f"_img-{image_init}_lat-{latent_init}.gif")
    FuncAnimation(fig, update, frames=len(frames)).save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return frames, frame_labels, energies, path

In [ ]:
digits = [1, 8, 4]                     # the sequence of clamped labels
steps_per_clamp = [2000, 10000, 10000]  # steps for each clamp (or a single int for all)
image_init = "noise"                 # "topdown" or "noise"
latent_init = "bottomup"                # "topdown" or "bottomup"
# For true generation from noise use image_init="noise", latent_init="bottomup",
# and give the first clamp ~6000 steps.

single_frames, single_labels, single_energies, single_gif = make_single_digit_gif(
    gen_model, digits, steps_per_clamp, lr_x=0.02,
    image_init=image_init, latent_init=latent_init, record_every=20, fps=25)

print(len(single_frames), "frames ->", single_gif)
display(Image(filename=str(single_gif)))   # shows the animated GIF inline